In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 11.8 MB/s eta 0:00:00


In [4]:
import anthropic
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "hello",
        }
    ],
)
print(message.content)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CYs2D1yWQ84TVaxeQxpMz'}

In [ ]:
import kagglehub

path = kagglehub.dataset_download("dhrubangtalukdar/qs-world-university-rankings-2026-top-1500")

print("Path to dataset files:", path)

In [ ]:
import os

print(os.listdir(path))

In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(path, "2026_QS_World University_Rankings.csv"))
df.head()

In [ ]:
df.columns

In [ ]:
import pandas as pd

selected_cols = [
    "Rank",
    "Name",
    "Country/Territory",
    "Region",
    "Overall SCORE",
    "Academic Reputation SCORE",
    "Employer Reputation SCORE",
    "Research",
    "Focus",
    "Size",
    "Status"
]

df = df[selected_cols].copy()

df = df.rename(columns={
    "Rank": "ranking",
    "Name": "university_name",
    "Country/Territory": "country",
    "Overall SCORE": "overall_score",
    "Academic Reputation SCORE": "academic_reputation",
    "Employer Reputation SCORE": "employer_reputation"
})

df.head()

In [ ]:
europe = [
    "United Kingdom", "Germany", "France", "Switzerland",
    "Netherlands", "Sweden", "Italy", "Spain",
    "Denmark", "Belgium", "Austria", "Finland",
    "Norway", "Ireland"
]

df = df[df["country"].isin(europe)].reset_index(drop=True)
df.head(30)

In [ ]:
df["combined_text"] = (
    "University name: " + df["university_name"].astype(str) + ". " +
    "Located in " + df["country"].astype(str) + ", region: " + df["Region"].astype(str) + ". " +
    "Global ranking: " + df["ranking"].astype(str) + ". " +
    "Overall score: " + df["overall_score"].astype(str) + ". " +
    "Academic reputation score: " + df["academic_reputation"].astype(str) + ". " +
    "Employer reputation score: " + df["employer_reputation"].astype(str) + ". " +
    "Research intensity: " + df["Research"].astype(str) + ". " +
    "Institution focus: " + df["Focus"].astype(str) + ". " +
    "University size: " + df["Size"].astype(str) + ". " +
    "Status: " + df["Status"].astype(str) + "."
)
df.head()

In [ ]:
df.to_csv("universities_base_europe.csv", index=False)

In [ ]:
print(os.listdir())

In [ ]:
df

In [ ]:
# df = pd.read_csv("universities_base_europe.csv")

chunks = df["combined_text"].tolist()
print(chunks)
print(f"Total chunks: {len(chunks)}")
print(f"\nExample chunk:\n{chunks[0]}")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks, show_progress_bar=True)
embeddings = np.array(embeddings)

print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
import numpy as np
from numpy import linalg as LA
normalized_embeddings = embeddings / np.linalg.norm(embeddings, axis = 1).reshape(-1, 1)
print(normalized_embeddings)

In [ ]:
def search(query, top_k=3):
    query_embedding = model.encode([query])

    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    similarities = normalized_embeddings @ query_embedding.T

    top_indices = np.argsort(similarities.flatten())[-top_k:][::-1]

    return [chunks[i] for i in top_indices]

In [ ]:
query = input()
results = search(query)
context = "Here is some context information:"
for r in results:
    print(r)
    context = context + "\n" + r
    print("---")
context = context + "\n" + f"Based on the above, please answer: {query}"


In [ ]:
import anthropic
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": context,
        }
    ],
)
print(message.content[0].text)